# Pilot 4 · rank-1 GRPO arm

**Implemented: preparation, sampling, offline validation, GPU preflight and protocol freeze (tickets 01–05).** Scientific training/recovery is now implemented; evaluation and measurement remain pending. Full training starts only in section 13 after the GPU preflight and reviewed freeze pass.

Pilot 2 trained rank-1 SFT on 512 GSM8K problems. Pilot 4 reuses those problems and optimizer prompt order, changing the objective to group-normalized reward training.

Sections 1–3 are CPU-only and verify saved Drive evidence. Sections 4–6 require one GPU and sample the untuned FP32 model. Section 8 uses tiny real models on CPU with no model download. It can run after dependencies are installed, without waiting for GPU sampling. Sections 9–12 resolve settings, collect real GPU preflight evidence and freeze the reviewed protocol. FP32 for both arms is approved. Historical accuracy 0.640 is contextual, not a correctness gate.


In [ ]:
from pathlib import Path
import json, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
HARNESS_COMMIT = '87489fa674ca5f46620da90476803cf4852385f3'
REPO_DIR = Path('/content/spar-pilot')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts')
SFT_PLAN = 'pilot2-sft-fp32-l4-batch8-v1' # @param {type:"string"}
MEASUREMENT_PLAN = 'pilot3-inputs-v1' # @param {type:"string"}
PLAN_NAME = 'pilot4-grpo-fp32-v1' # @param {type:"string"}
SFT_CONFIG = ARTIFACT_ROOT / 'plans' / SFT_PLAN / 'sft.config.json'
MEASUREMENT_CONFIG = ARTIFACT_ROOT / 'plans' / MEASUREMENT_PLAN / 'activation.prepared.json'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
assert not dirty, 'Preserve local repository edits before changing the checkout.'
subprocess.run(['git', 'fetch', 'origin', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
for source_path in [SFT_CONFIG, MEASUREMENT_CONFIG]:
    print(source_path, '| exists:', source_path.is_file())
    assert source_path.is_file(), 'Locate the saved source config in Drive before proceeding.'


## 2. Verify sources and prepare the plan

This calls the same public command tested offline. It verifies completed SFT checkpoints, frozen-base evidence, original training/evaluation data, actual saved prompt order and Pilot 3 token/mask hashes. It reuses those files rather than resampling or tokenizing again.

**Expected:** a prepared config, matching audit, metadata and completion marker. Repeating this cell revalidates/reuses the preparation. An incompatible source or corrupt artifact stops the command. Prepared source evidence does not mean training has run or the scientific protocol is frozen.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-prepare',
                '--sft-config', str(SFT_CONFIG), '--measurement-config', str(MEASUREMENT_CONFIG),
                '--name', PLAN_NAME, '--output-root', str(ARTIFACT_ROOT)],
               cwd=REPO_DIR, check=True)
PREPARED_PATH = ARTIFACT_ROOT / 'plans' / PLAN_NAME / 'grpo.prepared.json'


## 3. Read the matching audit

Review the identities and intended matches: 512 training problems, 150 held-out items, 150 GSM8K + 150 FineWeb measurement sequences, FP32 and rank-1 adapters. The saved SFT optimizer order is reused; the selected source indices alone do not specify training order.

The unresolved table is intentional. It identifies the scorer confirmation, exact verified loss, baseline subset/generation policy, final completion cap, monitor actions and preregistration. Later tickets resolve them before a full run. Historical aggregate results are not tests of this preparation.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-audit',
                '--config', str(PREPARED_PATH), '--output-root', str(ARTIFACT_ROOT)],
               cwd=REPO_DIR, check=True)
PLAN = json.loads(PREPARED_PATH.read_text())
print('Prepared config:', PREPARED_PATH)
print('Audit:', ARTIFACT_ROOT / PLAN['audit_path'])
print('Pending choices:', ', '.join(PLAN['pending']))
print('Next stage: untuned baseline sampling after its settings are explicitly resolved.')


## 4. Prepare the GPU sampling environment

Only sampling needs the GPU; source preparation above is CPU-only. Install the same pinned libraries used by the earlier pilots. If Colab requests a restart, restart and restore sections 1–3. This stage uses an untuned FP32 model on one GPU, not a hardware-specific T4/L4 code path.


In [ ]:
from pilot_eval.sft_backend import PINS
subprocess.run([sys.executable, '-m', 'pip', 'install', *[f'{k}=={v}' for k, v in PINS.items()]], check=True)


## 5. Supply explicit sampling choices

This is **128 problems from the 512 training problems**, eight draws each. It is neither the 150 held-out behavioural evaluation nor X1's deferred full 512-problem sampling baseline.

Create a JSON settings file in Drive. Required keys: `subset_ids` (128 unique training IDs in the chosen order), `scorer` (`gsm8k-flexible-v3`), `seed`, `groups_per_batch`, `max_new_tokens`, `temperature` (1.0), `top_p`, and `top_k`. No values are selected silently. One group is eight completions: `groups_per_batch=1` means an inference batch of eight, not one.

Choose the provisional cap, filters, batch policy and subset before running. The chosen sampling seed is distinct from cohort selection; record how you selected the subset in your notes. The command freezes all IDs/settings/batch seeds. Keep the same name/settings to resume; use a new name for a longer-cap extension.


In [ ]:
SAMPLING_SETTINGS_PATH = '' # @param {type:"string"}
BASELINE_NAME = 'pilot4-untuned-sampling-v1' # @param {type:"string"}
assert SAMPLING_SETTINGS_PATH.strip(), 'Supply the explicit reviewed settings JSON path first.'
SAMPLING_SETTINGS_PATH = Path(SAMPLING_SETTINGS_PATH)
assert SAMPLING_SETTINGS_PATH.is_file(), 'Sampling settings file is missing.'
print(json.dumps(json.loads(SAMPLING_SETTINGS_PATH.read_text()), indent=2))


## 6. Sample and resume the untuned baseline

The model loads once and batches prompts with eight draws per group. Each completed batch saves responses and a hash marker; a repeat verifies saved batches and generates only missing ones. An interrupted, unsealed batch is redone with its frozen seed. Settings/code/runtime mismatches stop reuse. Batch completion messages are work counters; this is not token-by-token progress. The subprocess exits on failure instead of polling a dead process.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-baseline',
                '--config', str(PREPARED_PATH), '--settings', str(SAMPLING_SETTINGS_PATH),
                '--name', BASELINE_NAME, '--output-root', str(ARTIFACT_ROOT)], cwd=REPO_DIR, check=True)
BASELINE_CONFIG = json.loads((ARTIFACT_ROOT / 'plans' / BASELINE_NAME / 'baseline.config.json').read_text())
BASELINE_DIR = ARTIFACT_ROOT / BASELINE_CONFIG['run_path']
BASELINE_SUMMARY = json.loads((BASELINE_DIR / 'results/results.json').read_text())
print(json.dumps({k:v for k,v in BASELINE_SUMMARY.items() if k != 'groups'}, indent=2))
print('Every draw:', BASELINE_DIR / 'results/responses.jsonl')
print('Per-group correct counts:', BASELINE_DIR / 'results/results.json')


## 7. Review the evidence before choosing the final cap

Read cap fraction, length percentiles, per-group correct counts and dead-group fraction. A dead group has all eight rewards equal (all correct or all incorrect), so it provides no within-group learning signal. Caps receive zero reward even if text contains an answer; this can encourage shorter responses.

If `p99_censored` is true, this run cannot justify the final cap. Preserve it and run an explicitly named extension with a larger provisional cap. Even an uncensored percentile does not guarantee later training responses will fit: the later protocol freeze must explicitly choose the final cap and cap-fraction monitoring policy. No training starts from this notebook stage yet.


## 8. Validate the GRPO implementation on CPU

This checks the training software before a scientific GPU run. It creates tiny random Qwen models locally; no pretrained model or dataset is downloaded. First it compares real TRL loss/LoRA gradients with an independent calculation and checks memory microbatch equivalence, padding, capped/dead groups and Adam momentum. Then it runs 20-step positive and reversed-sign learning controls at seeds 42 and 43.

The versioned learning gate compares steps 16–20 with the untouched policy's exact expected reward (50% in this binary task). Each sign must change reward by at least 20 points in each seed. Earlier failed fixtures remain preserved. The toy learning rate is distinct from the scientific rate.

Expected: `passed: true`, four passing learning controls, and saved configs, responses, per-step logs and initial/final adapters. Rerunning verifies and reuses sealed evidence. A failure stops the workflow; do not adjust a setting under the same run name. This does not authorize full training: GPU preflight and protocol freeze are still required.


In [ ]:
CONTROL_NAME = 'tiny-grpo-controls-v3-preflight-v1'
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-controls',
                '--name', CONTROL_NAME, '--output-root', str(ARTIFACT_ROOT)],
               cwd=REPO_DIR, check=True)
CONTROL_DIR = ARTIFACT_ROOT / 'runs/diagnostics/pilot-4' / CONTROL_NAME
CONTROL_RESULTS = json.loads((CONTROL_DIR / 'results/results.json').read_text())
assert CONTROL_RESULTS['passed'] and CONTROL_RESULTS['algorithm_checks']['passed']
print('Algorithm checks:', CONTROL_RESULTS['algorithm_checks']['checks'])
for row in CONTROL_RESULTS['runs']:
    print('Seed:', row['seed'], 'sign:', row['sign'], 'baseline:', row['baseline_reward'],
          'late reward:', row['late_reward'], 'signed change:', row['signed_change'], 'passed:', row['passed'])
print('Saved evidence:', CONTROL_DIR)


## 9. Resolve preflight settings explicitly

These are decisions, not measured results. Fill every `None` after reviewing the matching audit and sampling evidence. The software will not silently pick them.

- `scorer`: `gsm8k-flexible-v3`; `loss`: `dapo` uses the validated full-window completion-token normalization.
- `cap_extra_tokens`: positive integer added to the ceiling of the uncensored baseline p99. Review the resulting cap before running. This is an explicit cap rule; another rule requires a protocol amendment.
- `generation_groups` and `backward_groups`: 1, 2, 4 or 8. One group contains eight completions; these change memory use while preserving eight prompts per optimizer update.
- `evaluation_batch_size`: positive integer; `gradient_checkpointing`: true/false. Checkpointing saves training activations at the cost of recomputation.
- `diagnostic_seed`: nonnegative integer for disposable trials; `memory_margin_gib`: positive free-memory reserve required in the longest-input stress test.

Filters are read from the saved sampling policy, so baseline and training use the same distribution. Changing the policy requires another named baseline. Changed settings/code require a new preflight name; preserve earlier evidence.


In [ ]:
BASELINE_CONFIG_PATH = ARTIFACT_ROOT / 'plans' / BASELINE_NAME / 'baseline.config.json'
PREFLIGHT_NAME = PLAN_NAME + '-preflight-v1'
FROZEN_NAME = PLAN_NAME + '-frozen-v1'
PREFLIGHT_SETTINGS = {
    'scorer': None, 'loss': None,
    'cap_rule': {'method': 'ceil-p99-plus', 'extra_tokens': None},
    'generation_groups': None, 'backward_groups': None,
    'evaluation_batch_size': None, 'gradient_checkpointing': None,
    'diagnostic_seed': None, 'memory_margin_gib': None,
    'top_p': BASELINE_CONFIG['settings']['top_p'],
    'top_k': BASELINE_CONFIG['settings']['top_k'],
}
assert all(v is not None for v in PREFLIGHT_SETTINGS.values()) and PREFLIGHT_SETTINGS['cap_rule']['extra_tokens'] is not None, 'Resolve the choices above first.'
from pilot_eval.workflow import _save_frozen
PREFLIGHT_SETTINGS_PATH = ARTIFACT_ROOT / 'notebook' / PREFLIGHT_NAME / 'settings.json'
PREFLIGHT_SETTINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
_save_frozen(PREFLIGHT_SETTINGS_PATH, PREFLIGHT_SETTINGS)
print('Explicit settings:', json.dumps(PREFLIGHT_SETTINGS, indent=2))


## 10. Collect disposable GPU evidence

This loads the real pinned model and validates the 196 rank-1 adapters. It then:
1. Compares untuned and zero-adapter greedy token sequences on all 150 held-out items.
2. Runs two fresh, identical two-step trials on 16 training problems. Saves every sampled response, loss, pre-clip gradient, adapter/optimizer/RNG state and frozen-weight hashes; verifies exact repeatability.
3. Generates 64 full-cap completions from the eight longest training prompts and runs one backward/update capacity probe. This synthetic stress test uses artificial rewards solely to exercise memory; its state is discarded.

Saved units are verified before reuse after interruption. A failed gate stops; no full scientific training starts here. Timings include hash/transfer/save overhead for disposable trials. Projections describe training and behavioral evaluation; later activation-measurement costs remain unmeasured. The command prints progress; a successful notebook process alone is not a training result.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-preflight',
                '--config', str(PREPARED_PATH), '--baseline-config', str(BASELINE_CONFIG_PATH),
                '--controls-name', CONTROL_NAME, '--settings', str(PREFLIGHT_SETTINGS_PATH),
                '--name', PREFLIGHT_NAME, '--output-root', str(ARTIFACT_ROOT)], cwd=REPO_DIR, check=True)
PREFLIGHT_PATH = ARTIFACT_ROOT / 'plans' / PREFLIGHT_NAME / 'grpo.preflight.json'
PREFLIGHT_CONFIG = json.loads(PREFLIGHT_PATH.read_text())
PREFLIGHT_DIR = ARTIFACT_ROOT / PREFLIGHT_CONFIG['run_path']
PREFLIGHT_RESULTS = json.loads((PREFLIGHT_DIR / 'results/results.json').read_text())
print(json.dumps(PREFLIGHT_RESULTS, indent=2))
print('Saved GPU evidence:', PREFLIGHT_DIR)


## 11. Review evidence and write the preregistration

Check zero-adapter identity, both repeated trials, finite gradients, frozen weights and full-cap memory headroom. Compare cost projections with your available GPU budget. Inspect saved responses as well as summary numbers.

Fill the review below. The non-inferiority margin is a fraction of accuracy (0.10 means ten percentage points); it has no approved default. Monitor thresholds/actions also require explicit choices: actions are `flag`, `pause`, or `stop`. State your prediction and what result would count against it. The timestamp is recorded when you save this user-authored review. This records your decision before scientific training, rather than inventing a prediction after results.


In [ ]:
PREFLIGHT_REVIEWED = False
REVIEW_NOTES = ''
PREREG_AUTHOR = ''
PREREG_PREDICTION = ''
PREREG_FALSIFIER = ''
NONINFERIORITY_MARGIN = None
MONITORS = {
    'cap_fraction': {'threshold': None, 'action': None},
    'dead_group_fraction': {'threshold': None, 'action': None},
    'length_ratio_change_8_steps': {'threshold': None, 'action': None},
}
assert PREFLIGHT_REVIEWED and all(s.strip() for s in [REVIEW_NOTES, PREREG_AUTHOR, PREREG_PREDICTION, PREREG_FALSIFIER]), 'Review the evidence and write the preregistration first.'
assert NONINFERIORITY_MARGIN is not None and all(v is not None for m in MONITORS.values() for v in m.values()), 'Resolve the margin and monitor policies.'
from datetime import datetime, timezone
REVIEW_PATH = ARTIFACT_ROOT / 'notebook' / FROZEN_NAME / 'review.json'
REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
REVIEW_VALUES = dict(reviewed=True, notes=REVIEW_NOTES, margin=NONINFERIORITY_MARGIN,
    preregistration=dict(author=PREREG_AUTHOR, prediction=PREREG_PREDICTION, falsifier=PREREG_FALSIFIER), monitors=MONITORS)
if REVIEW_PATH.exists():
    previous = json.loads(REVIEW_PATH.read_text())
    REVIEW_VALUES['preregistration']['written_at'] = previous['preregistration']['written_at']
else:
    REVIEW_VALUES['preregistration']['written_at'] = datetime.now(timezone.utc).isoformat()
_save_frozen(REVIEW_PATH, REVIEW_VALUES)
print('Dated review:', REVIEW_PATH)


## 12. Freeze and verify the execution protocol

This verifies the saved evidence again and saves an immutable plan containing ordered prompt groups, precision, runtime, optimizer, scorer, cap, batching, review and source hashes. It rejects unresolved, altered or unreviewed evidence. Rerunning the same review verifies/reuses it; amendments use a new name.

Expected: `state: frozen`. This completes the preflight handoff. The 64-step scientific trainer follows in section 13; this cell does not run it. Recovery for that trainer will be from sealed checkpoints 0, 8, 16, 32 and 64, so up to 32 steps may need repeating.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-freeze',
                '--config', str(PREFLIGHT_PATH), '--review', str(REVIEW_PATH),
                '--name', FROZEN_NAME, '--output-root', str(ARTIFACT_ROOT)], cwd=REPO_DIR, check=True)
FROZEN_PATH = ARTIFACT_ROOT / 'plans' / FROZEN_NAME / 'grpo.frozen.json'
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-check-ready',
                '--config', str(FROZEN_PATH), '--output-root', str(ARTIFACT_ROOT)], cwd=REPO_DIR, check=True)
FROZEN = json.loads(FROZEN_PATH.read_text())
print('Protocol state:', FROZEN['state'])
print('Verified execution plan:', FROZEN_PATH)


## 13. Train or recover the scientific GRPO arm

Read the [Ticket 5 diagram companion](https://github.com/mahadikprasad15/spar-pilot/blob/main/output/learning/pilot4-ticket5-explained.html). This starts from a fresh untuned model with a zero adapter, not from preflight's disposable updates. It loads one model and follows the frozen generation/backward batches. Eight problems × eight draws form each optimizer update; updates run sequentially because each samples from the preceding policy.

Expected totals: 64 optimizer updates, 512 distinct training problems, 4,096 accepted draws, checkpoints 0/8/16/32/64. Progress distinguishes completed updates/prompts/draws from the latest sealed recovery position. Every draw and step diagnostic is saved. The approved length monitor compares current mean completion tokens with the mean eight steps earlier, using absolute fractional change; it starts at step 9 and is undefined for a zero previous mean.

Rerunning this cell restores only the latest verified sealed checkpoint. Later incomplete attempts remain diagnostic files and are excluded from accepted history. Up to 32 updates may need repeating. Completed runs are verified/reused without loading the GPU model. Do not edit the plan or batch sizes under the same name.

A paused/stopped monitor run needs inspection: repeating it keeps the same policy and can trigger the same monitor again. No silent bypass is offered. Numerical/integrity failures stop. The child process is waited for directly; its actual nonzero exit raises an error rather than polling a stale PID forever. Do not launch another GPU stage at the same time.


In [ ]:
TRAIN_RUN_NAME = PLAN_NAME + '-arm-v1' # @param {type:"string"}
TRAIN_PROCESS = subprocess.run([sys.executable, '-u', '-m', 'pilot_eval', 'grpo-train',
                '--config', str(FROZEN_PATH), '--name', TRAIN_RUN_NAME,
                '--length-change-definition', 'step-lag-8',
                '--output-root', str(ARTIFACT_ROOT)], cwd=REPO_DIR, check=False)
TRAINING_CONFIG_PATH = ARTIFACT_ROOT / 'plans' / TRAIN_RUN_NAME / 'grpo.training.json'
if TRAIN_PROCESS.returncode:
    if TRAINING_CONFIG_PATH.exists():
        from pilot_eval.grpo_prepare import _within
        from pilot_eval.training import run_lock, verified_checkpoints
        from pilot_eval.run import _write_state
        failed_directory = _within(ARTIFACT_ROOT.resolve(), json.loads(TRAINING_CONFIG_PATH.read_text())['run_path'])
        with run_lock(failed_directory):
            status_path = failed_directory / 'meta/status.json'
            status = json.loads(status_path.read_text()) if status_path.exists() else {}
            if status.get('state') == 'running':
                sealed = max(verified_checkpoints(failed_directory), default=0)
                _write_state(failed_directory, 'failed', sealed, 64,
                             f'Training child exited with code {TRAIN_PROCESS.returncode}; unsealed attempt progress is excluded.')
    raise RuntimeError(f'Training process exited with exit code {TRAIN_PROCESS.returncode}. Inspect saved status and attempts before resuming.')
TRAINING_CONFIG = json.loads(TRAINING_CONFIG_PATH.read_text())
TRAINING_DIR = ARTIFACT_ROOT / TRAINING_CONFIG['run_path']
TRAINING_RESULTS = json.loads((TRAINING_DIR / 'results/results.json').read_text())
print(json.dumps(TRAINING_RESULTS, indent=2))
print('Accepted training draws:', TRAINING_DIR / 'results/responses.jsonl')
print('Step diagnostics:', TRAINING_DIR / 'results/steps.jsonl')
print('Checkpoint states:', TRAINING_DIR / 'checkpoints')


## 14. Verify saved training completion (CPU only)

This verifies source/code identities, result hashes, all five checkpoint states, ordered groups and exactly 4,096 accepted draw IDs. It loads no model. Expected: 64 steps, 512 prompts, 4,096 completions and unchanged frozen base evidence. Passing establishes training completion, not held-out performance or scientific success.

After this verification you can disconnect the GPU if stopping here. Sections 15–16 add held-out greedy/sampled evaluation; Ticket 7 adds fixed-token internal measurements. The config below is the source handoff for those stages. Keep the complete artifact tree, including checkpoint-referenced attempt records, rather than exporting only the adapter files.


In [ ]:
subprocess.run([sys.executable, '-u', '-m', 'pilot_eval', 'grpo-verify-training',
                '--config', str(TRAINING_CONFIG_PATH), '--output-root', str(ARTIFACT_ROOT)],
               cwd=REPO_DIR, check=True)
print('Verified training source:', TRAINING_CONFIG_PATH)
print('Training records and all resume states are saved in Drive:', TRAINING_DIR)


## 15. Evaluate the verified GRPO checkpoints (GPU)

This runs seven inference settings on the identical 150 held-out questions: greedy at 0/8/16/32/64 (750 responses), and temperature-1 sampling at 0/64 with eight draws per item (2,400 responses). Total: 3,150 saved responses. It uses the frozen final cap, FP32 runtime, prompts and batch settings from preflight. Greedy batches contain `evaluation_batch_size` items; sampled batches contain `generation_groups` items × eight completions.

One model is loaded; checkpoints run sequentially on the single GPU. Each batch is reseeded with `42 + ordered batch index`, using the same schedule across checkpoints. Completed sealed batches are verified and reused; an interrupted unsealed batch is repeated. Changing batch membership, precision, code, filters or source artifacts is rejected under this name. The command waits for its actual child process exit; a live PID is not completion evidence.

Expected: 750 greedy + 2,400 sampled responses, both scorers, cap/end metadata and raw token IDs. This step generates outputs; section 16 computes the behavioural comparison. Existing SFT responses are not regenerated.


In [ ]:
BEHAVIOUR_NAME = PLAN_NAME + '-behaviour-v1' # @param {type:"string"}
subprocess.run([sys.executable, '-u', '-m', 'pilot_eval', 'grpo-evaluate',
                '--config', str(TRAINING_CONFIG_PATH), '--name', BEHAVIOUR_NAME,
                '--output-root', str(ARTIFACT_ROOT)], cwd=REPO_DIR, check=True)
BEHAVIOUR_CONFIG_PATH = ARTIFACT_ROOT / 'plans' / BEHAVIOUR_NAME / 'grpo.evaluation.json'
BEHAVIOUR_CONFIG = json.loads(BEHAVIOUR_CONFIG_PATH.read_text())
BEHAVIOUR_DIR = ARTIFACT_ROOT / BEHAVIOUR_CONFIG['run_path']
print('Saved held-out continuations:', BEHAVIOUR_DIR / 'results/responses.jsonl')
print('Frozen inference plan:', BEHAVIOUR_CONFIG_PATH)


## 16. Read the paired behavioural report (CPU only)

This verifies every sealed inference batch, joins checkpoints by item ID, and reports strict/flexible-v3 accuracy, cap/invalid rates and response-length distributions. Sampled accuracy averages all eight draw rewards per question, then all 150 questions; it is not pass-at-eight.

The paired bootstrap draws 10,000 resamples of 150 item IDs with seed 42. Each item carries both checkpoint records and all its sampled draws. The drop is checkpoint 0 minus checkpoint 64: positive means final accuracy is lower. Non-inferiority passes only when the upper 95% percentile interval is strictly below your frozen margin. A failed gate alone is inconclusive; harm beyond the margin requires the lower bound above it. Positive-decline evidence is reported separately.

The interval describes this cohort and its observed draws, not variation across training seeds or repeated sampling on this exact fixed cohort. Identical seeds do not guarantee identical answers after training. The report saves bootstrap indices, item order and every paired response. No model is loaded, so after evaluation completes you can disconnect the GPU and run this later with the same artifact tree and code.


In [ ]:
BEHAVIOUR_REPORT_NAME = PLAN_NAME + '-behaviour-paired-v1' # @param {type:"string"}
subprocess.run([sys.executable, '-u', '-m', 'pilot_eval', 'grpo-behaviour-report',
                '--config', str(BEHAVIOUR_CONFIG_PATH), '--name', BEHAVIOUR_REPORT_NAME,
                '--output-root', str(ARTIFACT_ROOT)], cwd=REPO_DIR, check=True)
BEHAVIOUR_REPORT_DIR = ARTIFACT_ROOT / 'reports' / BEHAVIOUR_REPORT_NAME
BEHAVIOUR_RESULTS = json.loads((BEHAVIOUR_REPORT_DIR / 'results/results.json').read_text())
if 'display' not in globals():
    from IPython.display import Markdown, display
display(Markdown((BEHAVIOUR_REPORT_DIR / 'results/report.md').read_text()))
print('Structured outcomes:', BEHAVIOUR_REPORT_DIR / 'results/results.json')
print('All paired responses:', BEHAVIOUR_REPORT_DIR / 'results/paired-items.jsonl')
print('Bootstrap resampling identities:', BEHAVIOUR_REPORT_DIR / 'results/bootstrap-indices.json')
